In [44]:
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials
from array import array
import os
from PIL import Image
import sys
import time

'''
Authenticate
Authenticates your credentials and creates a client.
'''
subscription_key = "DDSKJoNuiIOS14PRv1QH0KR5oGYxYKjuEOVFrD3Sk8AJGEZh207aJQQJ99BCAC5RqLJXJ3w3AAAFACOG0EZm"
endpoint = "https://resourcebibi.cognitiveservices.azure.com/"
computervision_client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))
'''
END - Authenticate
'''


#img = open("test1.png", "rb")
#img = open("test2.jpeg", "rb")
img = open("imagine_grayscale.png", "rb")

read_response = computervision_client.read_in_stream(
    image=img,
    mode="Printed",
    raw=True
)
# print(read_response.as_dict())

operation_id = read_response.headers['Operation-Location'].split('/')[-1]
while True:
    read_result = computervision_client.get_read_result(operation_id)
    if read_result.status not in ['notStarted', 'running']:
        break
    time.sleep(1)

# Get detected text, line by line
def get_detected_text():
    result = []
    if read_result.status == OperationStatusCodes.succeeded:
        for text_result in read_result.analyze_result.read_results:
            for line in text_result.lines:
                result.append(line.text)
    return " ".join(result)


def get_text_corect():
    referinta = "Succes în rezolvarea tEMELOR la LABORA toarele de Inteligență Artificială!"
    return referinta


def procesare_imagine(cale_imagine):
    with open(cale_imagine, "rb") as img:
        read_response = computervision_client.read_in_stream(image=img, raw=True)

    operation_id = read_response.headers['Operation-Location'].split('/')[-1]

    max_wait_time = 30
    start_time = time.time()
    while True:
        read_result = computervision_client.get_read_result(operation_id)
        if read_result.status not in ['notStarted', 'running']:
            break
        if time.time() - start_time > max_wait_time:
            raise TimeoutError("Procesarea Azure OCR a durat prea mult timp!")
        time.sleep(1)

    return read_result


In [38]:
import Levenshtein

# ex 1. a)

# Distanta Levenshtein -> nr de operatii de adaugare stergere sau inlocuire de care e nev ca sa se ajunga la textul bun
def distanta_Levenshtein(corect, detectat):
    dis_levenshtein = Levenshtein.distance(corect, detectat)
    return dis_levenshtein


# ex 1. b)

# Distanta Hamming -> ia poz cu poz sirurile si returneaza nr de caracteze diferite dintre ele
def calculate_hamming(text_corect, text_detectat):
    if len(text_corect) != len(text_detectat):
        raise ValueError("Sirurile nu au aceeasi lungime")
    return Levenshtein.hamming(text_corect, text_detectat)


# 0 -> diferite, 1 -> identice, cu cat mai mult cu atat mai bine
#j(a, b) = A ∩ B / A ∪ B
def jaccard(text_corect, text_detectat):
    multime_cuv_corecte = set(text_corect.split())
    multime_cuv_detectate = set(text_detectat.split())

    intersectie = len(multime_cuv_corecte.intersection(multime_cuv_detectate))
    reuniune = len(multime_cuv_corecte.union(multime_cuv_detectate))

    return intersectie / reuniune if reuniune != 0 else 0


In [39]:
import Levenshtein

# Character error rate
def calculate_cer(text_corect, text_detectat):
    #(carac inserate + carac gresite + carac inlocuite)/ nr caractere

    char_errors = Levenshtein.distance(text_corect, text_detectat)
    total_chars = len(text_corect)

    return char_errors / total_chars if total_chars != 0 else 0


# Word Error Rate
def calculate_wer(text_corect, text_detectat):
    #(cuv inserate + carac cuv + cuv inlocuite)/ nr cuv bune

    cuv_bune = text_corect.split()
    cuv_detectate = text_detectat.split()
    return Levenshtein.distance(cuv_bune, cuv_detectate) / len(cuv_bune)


In [40]:
import pandas as pd
from PIL import Image

# localizare cuvinte
def localizare_locatie_cuvinte(cale_imagine, rezultat):
    imagine = Image.open(cale_imagine)
    img_width, img_height = imagine.size
    data = []

    for text_result in rezultat.analyze_result.read_results:
        for linie in text_result.lines:
            for cuvant in linie.words:
                text = cuvant.text
                bbox = cuvant.bounding_box

                x_min = min(bbox[0], bbox[2], bbox[4], bbox[6])
                y_min = min(bbox[1], bbox[3], bbox[5], bbox[7])
                x_max = max(bbox[0], bbox[2], bbox[4], bbox[6])
                y_max = max(bbox[1], bbox[3], bbox[5], bbox[7])

                width = x_max - x_min
                height = y_max - y_min
                center_x, center_y = img_width // 2, img_height // 2

                vertical_position = "Top" if (y_min + height / 2) < center_y else "Bottom"
                horizontal_position = "Left" if (x_min + width / 2) < center_x else "Right"
                position = f"{vertical_position} {horizontal_position}"

                data.append({
                    "Cuvant": text,
                    "X_min": x_min,
                    "Y_min": y_min,
                    "X_max": x_max,
                    "Y_max": y_max,
                    "Latime": width,
                    "Inaltime": height,
                    "Pozitie": position
                })

    df = pd.DataFrame(data)
    return df



In [41]:
from PIL import Image, ImageFilter, ImageEnhance
import matplotlib.pyplot as plt

def filtrare_imagine(cale_imagine):
    imagine = Image.open(cale_imagine)
    # filtrul FIND_EDGES pentru detecția muchiilor
    edges_imagine = imagine.filter(ImageFilter.FIND_EDGES)

    var, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(imagine)
    axes[0].axis('off')
    axes[0].set_title("Before")

    axes[1].imshow(edges_imagine)
    axes[1].axis('off')
    axes[1].set_title("After")

    # Afișarea imaginilor
    plt.tight_layout()
    plt.show()
    return edges_imagine


"""
procesare mai buna
"""

from PIL import Image

ImG = Image.open('test2.jpeg')

#  în grayscale
gray_img = ImG.convert('L')

gray_img.show()

gray_img.save('imagine_grayscale.png')






In [45]:
def run():
    text_detectat = get_detected_text()
    text_corect = get_text_corect()

    print("Text corect: ", text_corect)
    print("Text detectat: ", text_detectat)
    print()

    dis_levenshtein = distanta_Levenshtein(text_corect, text_detectat)
    print("Distanta Levenshtein: ", dis_levenshtein)

    dis_hamming = calculate_hamming(text_corect, text_detectat)
    print("Distanta Hamming: ", dis_hamming)

    jac = jaccard(text_corect, text_detectat)
    print("Jaccard: ", jac)

    print("CER (Character error rate): ", calculate_cer(text_corect, text_detectat))
    print("WER (Word Error Rate): ", calculate_wer(text_corect, text_detectat))
    print()

    cale_img = "test2.jpeg"
    rezultat = procesare_imagine(cale_img)
    rezultat_df = localizare_locatie_cuvinte(cale_img, rezultat)
    print("\n Tabel cu locatia cuvintelor:")
    print(rezultat_df)


run()


Text corect:  Succes în rezolvarea tEMELOR la LABORA toarele de Inteligență Artificială!
Text detectat:  Lucces in resolvarea TEMELOR la LABORA toarele de Inteligenta Artificialà!

Distanta Levenshtein:  7
Distanta Hamming:  7
Jaccard:  0.25
CER (Character error rate):  0.0945945945945946
WER (Word Error Rate):  0.6


 Tabel cu locatia cuvintelor:
         Cuvant  X_min   Y_min   X_max   Y_max  Latime  Inaltime       Pozitie
0        Lucces   86.0   314.0   435.0   478.0   349.0     164.0      Top Left
1            in  476.0   312.0   651.0   472.0   175.0     160.0      Top Left
2    resolvarea  679.0   288.0  1328.0   465.0   649.0     177.0     Top Right
3       TEMELOR  142.0   588.0   739.0   727.0   597.0     139.0      Top Left
4            la  894.0   593.0  1046.0   725.0   152.0     132.0     Top Right
5        LABORA   79.0   916.0   400.0  1016.0   321.0     100.0      Top Left
6       toarele  417.0   916.0   754.0  1032.0   337.0     116.0      Top Left
7            de  8